_____

<table align="left" width=100%>
    <td>
        <div style="text-align: center;">
          <img src="../images/bar.png" alt="entidades financiadoras"/>
        </div>
    </td>
    <td>
        <p style="text-align: center; font-size:24px;"><b>Introduction to Data Science</b></p>
        <p style="text-align: center; font-size:18px;"><b>Master in Electrical and Computer Engineering</b></p>
        <p style="text-align: center; font-size:14px;"><b>Pedro Cardoso (pcardoso@ualg.pt)</b></p>
    </td>
</table>

_____

**Exercises:** Storing Time Series & Anomaly Detection — Portugal Mortality Data

# Exercises — Mortality Data: Loading, Storing, and Anomaly Detection

The dataset used in these exercises is located in `data/portugal_mortality/`. It contains **daily mortality counts recorded in Portugal**, obtained from the EEVV surveillance system ([evm.min-saude.pt](http://evm.min-saude.pt/)), covering the years **2014–2023**.

Two types of files are provided:

- **`causeXXXX.csv`** — daily number of deaths classified by cause of death: natural death (*Morte natural*), external cause (*Causa externa*), and under investigation (*Sujeito a investigação*).
- **`grupo_etarioXXXX.csv`** — daily number of deaths by age group.

> **Note:** From 2020 onwards the age-group files split the `> 75 anos` column into two columns: `75-84 anos` and `≥ 85 anos`. Your loading code must handle this schema change.

---
## Part 1 — Loading and Storing Data

## Exercise 1 — Load the cause-of-death data

Load all `causeXXXX.csv` files (2014–2023) from `data/portugal_mortality/` into a single pandas DataFrame.

Requirements:
- Each file contains dates in the format `Jan-01` (month-day). Combine the day–month string with the year encoded in the filename to build a proper `datetime` index (e.g. `2020-01-01`).
- Concatenate all files in **chronological order** and set the parsed date as the index.
- Column names contain Portuguese characters — read them as-is; do not rename.
- Display the first and last five rows of the resulting DataFrame.

**Hint:** `pd.to_datetime(str(year) + '-' + date_string, format='%Y-%b-%d')` — check `strptime` format codes for abbreviated month names.

## Exercise 2 — Inspect and describe the cause-of-death DataFrame

1. Display the data types of each column and check for missing values.
2. Compute and display summary statistics (mean, std, min, max, quartiles) for all columns.
3. How many days are recorded in total? Does this match the expected number of days between 1 January 2014 and 31 December 2023?

## Exercise 3 — Load the age-group data

Repeat the loading process for the `grupo_etarioXXXX.csv` files.

- Apply the same date-parsing strategy as in Exercise 1.
- The files for 2014–2019 contain one column for `> 75 anos`; from 2020 onwards this is split into `75-84 anos` and `≥ 85 anos`. Merge these into a common column `75+ anos` (sum of `75-84 anos` and `≥ 85 anos` for 2020–2023; rename `> 75 anos` for 2014–2019) so the final DataFrame has a consistent schema.
- Add a column `Total` containing the row-wise sum of all numeric age-group columns.
- Display the column names, shape, and the first few rows.

## Exercise 4 — Store and reload the data

Using the cause-of-death DataFrame built in Exercise 1:

1. Store it as a **CSV** file: `data/portugal_mortality/cause_all.csv`.
2. Store it as a **Parquet** file: `data/portugal_mortality/cause_all.parquet`.
3. Store it using **`pandas.HDFStore`**: `data/portugal_mortality/cause_all.h5`, under the key `/cause`.
4. Reload each file into a new DataFrame and verify that the index is parsed as `DatetimeIndex` and the values match the original.

## Exercise 5 — Compare storage formats

Compare the three formats saved in Exercise 4:

| Format  | File size (KB) | Load time (ms) |
|---------|---------------|----------------|
| CSV     | ?             | ?              |
| Parquet | ?             | ?              |
| HDF5    | ?             | ?              |

Measure file sizes with `os.path.getsize` and load times with `%timeit` (or `time.time`). Fill in the table and briefly comment on the trade-offs between the formats.

---
## Part 2 — Anomaly Detection

The following exercises use the cause-of-death DataFrame from Exercise 1. Unless stated otherwise, focus on the **`Morte natural`** (natural death) column.

## Exercise 6 — Visualise the time series

1. Plot `Morte natural` over the full period (2014–2023).
2. Overlay a **30-day rolling mean** on the same axes to highlight the long-term trend.
3. Plot the **annual total deaths** per year as a bar chart.

Comment on any periods that appear unusual (e.g. seasonal peaks, anomalous years).

## Exercise 7 — Distribution analysis

1. Plot a histogram of `Morte natural`. Does the distribution appear approximately normal?
2. Overlay a **KDE curve** on the histogram (use `density=True`).
3. Compute the **skewness** and **kurtosis** of the column.

Comment on whether the normality assumption required by Z-score-based methods seems appropriate for this dataset.

## Exercise 8 — Z-score method

Apply the Z-score method to detect anomalies in `Morte natural`:

1. Compute the Z-score for each day.
2. Flag days where $|Z| > 3$ as potential anomalies.
3. Plot the time series, marking the flagged days in red.
4. List the 10 days with the highest Z-scores. What events do these dates correspond to?

Repeat with threshold $|Z| > 2$ and comment on the change in the number of flagged days.

## Exercise 9 — Box-plot method

Apply the box-plot (IQR) method to detect outliers in `Morte natural`:

1. Compute $Q_1$, $Q_3$, and $\mathrm{IQR} = Q_3 - Q_1$.
2. Define outliers as points below $Q_1 - 1.5\,\mathrm{IQR}$ or above $Q_3 + 1.5\,\mathrm{IQR}$.
3. Plot a **box plot grouped by year**. Which years show the most variation?
4. Plot the time series with outliers highlighted.

How do the flagged days compare to those found by the Z-score method in Exercise 8?

## Exercise 10 — Modified Z-score method

The modified Z-score uses the **median absolute deviation (MAD)** and is more robust to extreme values:

$$M_i = \frac{0.6745\,(x_i - \tilde{x})}{\mathrm{MAD}}$$

where $\tilde{x}$ is the median and $\mathrm{MAD} = \mathrm{median}(|x_i - \tilde{x}|)$.

1. Implement the modified Z-score and apply it to `Morte natural`.
2. Flag days where $|M_i| > 3.5$ (the commonly used threshold).
3. Compare the flagged days with those from Exercises 8 and 9. Are the results consistent?
4. Repeat using `Causa externa` instead of `Morte natural`. Are the distributions and outliers different?

## Exercise 11 — Seasonal baseline and residual anomalies

A practical approach for seasonal data is to compare each day against a **historical seasonal baseline**.

1. Compute the **day-of-year average** of `Morte natural` using data from 2014 to 2019 (pre-COVID baseline).
2. Compute the **residual**: $r_t = x_t - \bar{x}_{\text{doy}(t)}$.
3. Apply the modified Z-score to the residuals. Flag days where $|M_i| > 3.5$.
4. Plot the residuals over time with flagged days highlighted.

Does this approach identify a cleaner set of anomalies compared to applying the modified Z-score directly to the raw values? Why?

## Exercise 12 — Isolation Forest (univariate)

Apply the Isolation Forest algorithm to the `Morte natural` column:

1. Build feature vectors using a **sliding window of 7 days** — each row is a vector of 7 consecutive daily values.
2. Train an `IsolationForest` with `contamination=0.05`.
3. Map predictions back to the original time series (use the **last day of each window** as the reference date).
4. Plot the time series with flagged anomalies highlighted.
5. Compare the detected anomalies with those from the statistical methods in Exercises 8–11.

**Hint:** Use `np.lib.stride_tricks.sliding_window_view` or a loop to build the window matrix.

## Exercise 13 — Isolation Forest (multivariate)

Isolation Forest naturally handles multiple features. Repeat Exercise 12 using **all three cause-of-death columns simultaneously** (`Morte natural`, `Causa externa`, `Sujeito a investigação`).

1. Build the sliding-window feature matrix using all three columns (window size = 7 days, giving a feature vector of length 21).
2. Train and apply the Isolation Forest (`contamination=0.05`).
3. Plot each column separately, marking the flagged anomalous windows.

Does the multivariate approach flag different events than the univariate approach in Exercise 12?

## Exercise 14 — Age-group analysis during anomalous periods

Using the age-group DataFrame from Exercise 3 and the anomaly flags produced by any of the methods above:

1. Extract the rows corresponding to **anomalous days**.
2. Compute the **average daily death count per age group** for anomalous days versus normal days.
3. Plot a **grouped bar chart** comparing the two averages.
4. Compute and plot the **percentage increase** in deaths per age group on anomalous days relative to the baseline.

Which age groups are most affected during anomalous periods? Can you identify a pattern related to known events (e.g. COVID-19 waves, heat waves)?